# EDA 03 — Limites Administratives des Arrondissements Parisiens
**Source** : Paris Open Data — API Explore v2.1  
**Bronze path** : `data/bronze/boundaries/part-0.parquet`  
**Scope** : 20 arrondissements de Paris

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `arrondissement` | int | Numéro (1-20) |
| `c_ar` | str | Code court |
| `c_arinsee` | str | Code INSEE (751xx) |
| `l_ar` | str | Label officiel |
| `surface_ha` | float | Superficie (m²) |
| `centroid_lat/lon` | float | Centroïde |
| `geometry_wkt` | str | Polygone WKT (EPSG:4326) |


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pd.set_option("display.max_columns", None)

BRONZE_ROOT = os.path.abspath("../../data/bronze")
path = f"{BRONZE_ROOT}/boundaries/part-0.parquet"

if os.path.exists(path):
    df = pd.read_parquet(path)
    print(f"Shape : {df.shape}")
else:
    print("⚠️  Fichier absent — démonstration impossible pour les géométries")
    df = pd.DataFrame()
df


In [ ]:
if not df.empty:
    # ── Superficie par arrondissement ───────────────────────────────────────
    df_sorted = df.sort_values("arrondissement")
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    surface_m2 = df_sorted["surface_ha"] / 1e6  # m² → km²
    colors = plt.cm.YlOrRd(surface_m2 / surface_m2.max())
    axes[0].bar(df_sorted["arrondissement"].astype(str), surface_m2, color=colors)
    axes[0].set_xlabel("Arrondissement")
    axes[0].set_ylabel("Superficie (km²)")
    axes[0].set_title("Superficie des arrondissements parisiens")
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    # Carte des centroides
    sc = axes[1].scatter(df_sorted["centroid_lon"], df_sorted["centroid_lat"],
                         c=surface_m2, cmap="YlOrRd", s=300, zorder=5)
    for _, row in df_sorted.iterrows():
        axes[1].annotate(str(row["arrondissement"]),
                         (row["centroid_lon"], row["centroid_lat"]),
                         ha="center", va="center", fontsize=7, fontweight="bold")
    plt.colorbar(sc, ax=axes[1], label="km²")
    axes[1].set_title("Centroides des arrondissements")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    
    plt.tight_layout()
    plt.show()


In [ ]:
if not df.empty:
    print("=== Statistiques géographiques ===")
    stats = df[["arrondissement","l_ar","surface_ha","centroid_lat","centroid_lon"]].copy()
    stats["surface_km2"] = (stats["surface_ha"] / 1e6).round(3)
    stats_sorted = stats.sort_values("surface_km2", ascending=False)
    print(f"Arrondissement le plus grand  : {stats_sorted.iloc[0]['l_ar']} ({stats_sorted.iloc[0]['surface_km2']:.2f} km²)")
    print(f"Arrondissement le plus petit  : {stats_sorted.iloc[-1]['l_ar']} ({stats_sorted.iloc[-1]['surface_km2']:.2f} km²)")
    print(f"Superficie totale Paris       : {stats['surface_km2'].sum():.2f} km²")
    display(stats[["arrondissement","l_ar","surface_km2"]].sort_values("arrondissement"))


In [ ]:
# ── Visualisation des polygones via Shapely ─────────────────────────────────
try:
    from shapely import wkt
    from shapely.geometry import mapping
    import matplotlib.cm as cm
    
    fig, ax = plt.subplots(figsize=(10, 12))
    cmap = cm.get_cmap("tab20")
    
    for i, row in df.iterrows():
        try:
            poly = wkt.loads(row["geometry_wkt"])
            x, y = poly.exterior.xy
            color = cmap(row["arrondissement"] / 20)
            ax.fill(x, y, alpha=0.6, color=color)
            ax.plot(x, y, color="white", linewidth=0.8)
            cx, cy = poly.centroid.x, poly.centroid.y
            ax.text(cx, cy, str(row["arrondissement"]), ha="center", va="center",
                    fontsize=8, fontweight="bold", color="white",
                    bbox=dict(boxstyle="round,pad=0.1", facecolor="black", alpha=0.4))
        except Exception:
            pass
    
    ax.set_title("Carte des 20 arrondissements parisiens", fontsize=14)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Shapely non disponible — pip install shapely pour la visualisation polygonale")


## Conclusions EDA
- Les 20 arrondissements couvrent ~105 km² avec une grande disparité de superficie (16e arr. ~7x plus grand que le 2e).
- Les géométries WKT sont propres et en projection EPSG:4326 — prêtes pour les jointures spatiales Silver.
- Le centroïde permet des calculs de distance rapides sans décodage des polygones.
